# FPIS Exploratory Data Analysis (EDA)

Notebook ini berfungsi sebagai walkthrough Langkah 1-3 dari pipeline FPIS.
Kita akan memuat data historis TLKM, mengecek metrik, dan melihat distribusi data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from pathlib import Path

plt.style.use('ggplot')
sns.set_palette('tab10')

# Gunakan sqlite connection yang ada jika pipeline sudah jalan
db_path = Path('../data/fpis.db')
if db_path.exists():
    conn = sqlite3.connect(db_path)
    df_q = pd.read_sql('SELECT * FROM preprocessed_quarterly', conn)
    df_a = pd.read_sql('SELECT * FROM preprocessed_annual', conn)
    conn.close()
else:
    print('Jalankan pipeline terlebih dahulu untuk generate database')
    # Fallback to local raw if needed
    df_q = pd.read_csv('../data/tlkm_quarterly_tidy.csv')

### 1. Cek Ketersediaan Data Kuartalan

In [ ]:
print(f"Total baris: {len(df_q)}")
print(f"Jumlah metrik unik: {df_q['metric_id'].nunique()}")

periods = df_q[['year', 'quarter']].drop_duplicates().sort_values(['year', 'quarter'])
print(f"\nPeriode ({len(periods)} kuartal):")
display(periods.head(3))
display(periods.tail(3))

### 2. Plot Revenue dan Net Income Historis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

rev = df_q[df_q['metric_id'] == 'revenue'].sort_values(['year', 'quarter'])
netinc = df_q[df_q['metric_id'] == 'netinc'].sort_values(['year', 'quarter'])

x_labels = [f"Q{q} '{str(y)[-2:]}" for y, q in zip(rev['year'], rev['quarter'])]

ax.plot(x_labels, rev['value'], marker='o', label='Revenue', linewidth=2)
ax.plot(x_labels, netinc['value'], marker='s', label='Net Income', linewidth=2)

ax.set_title('TLKM Revenue & Net Income (Miliar Rp)', fontsize=14)
ax.set_xticklabels(x_labels, rotation=45)
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

### 3. Distribusi Profitability Margin

In [ ]:
margins = ['grossMargin', 'operatingMargin', 'ebitdaMargin', 'profitMargin']
margin_data = df_q[df_q['metric_id'].isin(margins)].copy()

plt.figure(figsize=(10, 6))
sns.boxplot(data=margin_data, x='metric_id', y='value')
plt.title('Distribusi Profitability Margins')
plt.ylabel('Margin (Desimal)')
plt.show()